# Interpreting the ESO spectrum object-type output

This notebook reads the SQLite database produced by `eso-object-types run`. It explains how ESO spectra, catalog objects, object types, and positional links fit together.

Important scientific interpretation:

- A link means that a catalog object lies inside the cone used for an ESO spectrum.
- It does **not** mean that the object was detected in the spectrum or was the intended target.
- The prototype radius is `max(s_fov / 2, 1 arcsec)`. For production, the meaning of `s_fov` should be validated by instrument and replaced by `s_region` where appropriate.
- Only the primary SIMBAD type is stored.

In [ ]:
from pathlib import Path
import json
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)

def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root from the current directory")

REPO_ROOT = find_repository_root(Path.cwd().resolve())
DB_PATH = REPO_ROOT / "output" / "eso_object_types.sqlite"
if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Database not found at {DB_PATH}. Run `eso-object-types run` first."
    )

connection = sqlite3.connect(DB_PATH)
print(f"Database: {DB_PATH}")

## 1. Select a pipeline run

The database can contain several runs. By default, the notebook selects the latest one. Change `RUN_ID` below if you want to inspect an older run.

In [ ]:
runs = pd.read_sql_query(
    """
    SELECT run_id, started_at, finished_at, status, config_json, summary_json
    FROM pipeline_runs
    ORDER BY started_at DESC
    """,
    connection,
)
if runs.empty:
    raise RuntimeError("The database contains no pipeline runs")

display(runs[["run_id", "started_at", "finished_at", "status"]])
RUN_ID = runs.loc[0, "run_id"]
run_config = json.loads(runs.loc[0, "config_json"])
run_summary = json.loads(runs.loc[0, "summary_json"] or "{}")
print(f"Selected run: {RUN_ID}")
display(pd.Series(run_config, name="value").rename_axis("setting").to_frame())

## 2. Understand the relational model

The central relationship is:

`pipeline_runs -> run_observations -> observations -> observation_objects -> catalog_objects -> object_types`

- `observations` contains one row for each ESO data product, keyed by `eso_dp_id`.
- `run_observations` says which ESO rows belong to this run.
- `catalog_objects` stores each SIMBAD object once, keyed by `(catalog, catalog_object_id)`.
- `observation_objects` is the many-to-many link table. One spectrum can contain several objects, and one object can fall within several spectra.
- `object_types` avoids repeating type labels and descriptions for every object.
- `service_calls` records batching, attempts, timing, completion, and failures.

In [ ]:
service_calls = pd.read_sql_query(
    """
    SELECT service, batch_number, input_count, attempt_count, result_count,
           status, ROUND(elapsed_seconds, 3) AS elapsed_seconds,
           error_type, error_message
    FROM service_calls
    WHERE run_id = ?
    ORDER BY service, batch_number
    """,
    connection,
    params=(RUN_ID,),
)
display(service_calls)

## 3. Inspect the ESO spectrum sample

`search_radius_arcsec` is the actual matching radius. `s_fov_arcsec` is the full characteristic field of view reported by ESO. The order-10 HEALPix value is a spatial partition key for future scaling, not a match criterion.

In [ ]:
observations = pd.read_sql_query(
    """
    SELECT o.eso_dp_id, o.target_name, o.instrument_name,
           o.ra_deg, o.dec_deg,
           ROUND(o.s_fov_deg * 3600.0, 4) AS s_fov_arcsec,
           ROUND(o.search_radius_deg * 3600.0, 4) AS search_radius_arcsec,
           o.healpix_order10, o.s_region
    FROM observations AS o
    JOIN run_observations AS ro USING (eso_dp_id)
    WHERE ro.run_id = ?
    ORDER BY o.eso_dp_id
    """,
    connection,
    params=(RUN_ID,),
)
print(f"ESO spectra in run: {len(observations)}")
display(observations.head(10))

## 4. Inspect catalog matches

The catalog ID is the deduplication key within its provider: SIMBAD `oid`. `separation_arcsec` is calculated from the catalog coordinate and the ESO spectrum coordinate.

In [ ]:
matches = pd.read_sql_query(
    """
    SELECT oo.eso_dp_id, o.target_name AS eso_target_name,
           oo.catalog, oo.catalog_object_id,
           co.preferred_name AS catalog_name,
           co.primary_type_code, ot.type_label, ot.type_description,
           ROUND(oo.separation_arcsec, 5) AS separation_arcsec,
           ROUND(o.search_radius_deg * 3600.0, 5) AS search_radius_arcsec
    FROM observation_objects AS oo
    JOIN run_observations AS ro USING (eso_dp_id)
    JOIN observations AS o USING (eso_dp_id)
    JOIN catalog_objects AS co
      ON co.catalog = oo.catalog
     AND co.catalog_object_id = oo.catalog_object_id
    LEFT JOIN object_types AS ot
      ON ot.catalog = co.catalog
     AND ot.type_code = co.primary_type_code
    WHERE ro.run_id = ?
    ORDER BY oo.eso_dp_id, oo.catalog, oo.separation_arcsec
    """,
    connection,
    params=(RUN_ID,),
)
print(f"Object-to-spectrum links: {len(matches)}")
print(f"Unique catalog objects: {matches[["catalog", "catalog_object_id"]].drop_duplicates().shape[0]}")
display(matches.head(20))

## 5. Which object types occur most often?

These counts are link counts, not unique-object counts. An object appearing in several ESO spectra contributes one link to each spectrum.

In [ ]:
type_counts = pd.read_sql_query(
    """
    SELECT oo.catalog,
           COALESCE(co.primary_type_code, 'UNKNOWN') AS type_code,
           COALESCE(ot.type_label, co.primary_type_code, 'Unknown') AS type_label,
           COUNT(*) AS link_count,
           COUNT(DISTINCT oo.catalog_object_id) AS unique_objects,
           COUNT(DISTINCT oo.eso_dp_id) AS spectra
    FROM observation_objects AS oo
    JOIN run_observations AS ro USING (eso_dp_id)
    JOIN catalog_objects AS co
      ON co.catalog = oo.catalog
     AND co.catalog_object_id = oo.catalog_object_id
    LEFT JOIN object_types AS ot
      ON ot.catalog = co.catalog
     AND ot.type_code = co.primary_type_code
    WHERE ro.run_id = ?
    GROUP BY oo.catalog, COALESCE(co.primary_type_code, 'UNKNOWN')
    ORDER BY link_count DESC, oo.catalog, type_code
    """,
    connection,
    params=(RUN_ID,),
)
display(type_counts)

plot_data = type_counts.head(20).copy()
plot_data["display_type"] = plot_data["catalog"] + ":" + plot_data["type_code"]
ax = plot_data.sort_values("link_count").plot.barh(
    x="display_type", y="link_count", legend=False, figsize=(8, 5)
)
ax.set_xlabel("Observation-object links")
ax.set_ylabel("Catalog:type")
ax.set_title(f"Primary object types in {RUN_ID}")
plt.tight_layout()

## 6. Prototype an archive query by object type

Change `TYPE_CODE` to test the archive-style lookup. Use the previous table to find valid SIMBAD codes. `CATALOG` remains set to `simbad`.

In [ ]:
CATALOG = "simbad"
TYPE_CODE = "SN*"

spectra_by_type = pd.read_sql_query(
    """
    SELECT DISTINCT o.eso_dp_id, o.target_name, o.instrument_name,
           o.ra_deg, o.dec_deg,
           co.preferred_name AS matched_object,
           co.primary_type_code,
           ROUND(oo.separation_arcsec, 5) AS separation_arcsec
    FROM observations AS o
    JOIN run_observations AS ro USING (eso_dp_id)
    JOIN observation_objects AS oo USING (eso_dp_id)
    JOIN catalog_objects AS co
      ON co.catalog = oo.catalog
     AND co.catalog_object_id = oo.catalog_object_id
    WHERE ro.run_id = ?
      AND co.catalog = ?
      AND co.primary_type_code = ?
    ORDER BY o.eso_dp_id, oo.separation_arcsec
    """,
    connection,
    params=(RUN_ID, CATALOG, TYPE_CODE),
)
print(f"Spectra containing {CATALOG}:{TYPE_CODE}: {spectra_by_type['eso_dp_id'].nunique()}")
display(spectra_by_type)

## 7. A human-readable object list for each spectrum

This is close to what an archive results page could display. The relational tables remain preferable for storage because this combined string is only a presentation.

In [ ]:
objects_per_spectrum = pd.read_sql_query(
    """
    SELECT o.eso_dp_id, o.target_name,
           COUNT(oo.catalog_object_id) AS object_links,
           GROUP_CONCAT(
               oo.catalog || ':' || COALESCE(co.preferred_name, co.catalog_object_id)
               || ' [' || COALESCE(co.primary_type_code, 'UNKNOWN') || ']',
               '; '
           ) AS objects_in_cone
    FROM observations AS o
    JOIN run_observations AS ro USING (eso_dp_id)
    LEFT JOIN observation_objects AS oo USING (eso_dp_id)
    LEFT JOIN catalog_objects AS co
      ON co.catalog = oo.catalog
     AND co.catalog_object_id = oo.catalog_object_id
    WHERE ro.run_id = ?
    GROUP BY o.eso_dp_id, o.target_name
    ORDER BY object_links DESC, o.eso_dp_id
    """,
    connection,
    params=(RUN_ID,),
)
display(objects_per_spectrum)

## 8. One best object per spectrum

The best-object table gives one simple result per ESO spectrum. SIMBAD records remain available underneath it for provenance. Name or alias matches are preferred over distance, and confidence records how decisive the choice was.

In [ ]:
best_table_exists = connection.execute(
    "SELECT 1 FROM sqlite_master WHERE type = 'table' AND name = 'observation_best_objects'"
).fetchone()

if best_table_exists:
    best_objects = pd.read_sql_query(
        """
        SELECT bo.eso_dp_id, o.target_name, bo.best_object_name,
               bo.broad_category, bo.subcategory, bo.classification_detail,
               bo.confidence, bo.match_method, bo.separation_arcsec,
               bo.candidate_group_count, bo.supporting_catalogs,
               bo.raw_catalog_types
        FROM observation_best_objects AS bo
        JOIN observations AS o USING (eso_dp_id)
        WHERE bo.run_id = ?
        ORDER BY bo.confidence, bo.eso_dp_id
        """,
        connection,
        params=(RUN_ID,),
    )
    display(best_objects)

    category_counts = best_objects.groupby(
        ["broad_category", "subcategory"], dropna=False
    ).size().rename("spectra").sort_values(ascending=False).to_frame()
    confidence_counts = best_objects["confidence"].value_counts().rename("spectra").to_frame()
    display(category_counts)
    display(confidence_counts)

    plot_data = (
        best_objects["broad_category"]
        .fillna("Unknown")
        .value_counts()
        .sort_values()
        .rename_axis("broad_category")
        .reset_index(name="spectra")
    )
    ax = plot_data.plot.barh(
        x="broad_category",
        y="spectra",
        legend=False,
        figsize=(9, max(4, 0.55 * len(plot_data))),
        color="#4C78A8",
        edgecolor="white",
        linewidth=0.8,
    )
    ax.bar_label(ax.containers[0], fmt="{:,.0f}", padding=4, fontsize=9)
    ax.set_xlim(0, plot_data["spectra"].max() * 1.12)
    ax.set_xlabel("Number of spectra")
    ax.set_ylabel("Broad category")
    ax.set_title(f"Broad object categories in {RUN_ID}", pad=12, weight="bold")
    ax.xaxis.grid(True, linestyle="--", alpha=0.3)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
    plt.tight_layout()
    plt.show()

    review_examples = pd.read_sql_query(
        """
        SELECT bo.eso_dp_id, o.target_name, bo.best_object_name,
               bo.confidence, bo.match_method, bo.candidate_group_count,
               GROUP_CONCAT(
                   oo.catalog || ':' || COALESCE(co.preferred_name, co.catalog_object_id)
                   || ' [' || COALESCE(co.primary_type_code, 'UNKNOWN') || ']',
                   '; '
               ) AS all_positional_candidates
        FROM observation_best_objects AS bo
        JOIN observations AS o USING (eso_dp_id)
        LEFT JOIN observation_objects AS oo USING (eso_dp_id)
        LEFT JOIN catalog_objects AS co
          ON co.catalog = oo.catalog
         AND co.catalog_object_id = oo.catalog_object_id
        WHERE bo.run_id = ? AND bo.confidence IN ('low', 'none')
        GROUP BY bo.eso_dp_id, o.target_name, bo.best_object_name,
                 bo.confidence, bo.match_method, bo.candidate_group_count
        ORDER BY bo.candidate_group_count DESC, bo.eso_dp_id
        LIMIT 20
        """,
        connection,
        params=(RUN_ID,),
    )
    display(review_examples)
else:
    print("Best-object tables are not present. Run eso-object-types report to migrate and rebuild this run.")

## 9. Spectra with no catalog match

A zero-match spectrum is a valid result. It may reflect a small footprint, incomplete catalog coverage, coordinate differences, or an object missing from SIMBAD.

In [ ]:
unmatched = pd.read_sql_query(
    """
    SELECT o.eso_dp_id, o.target_name, o.instrument_name,
           o.ra_deg, o.dec_deg,
           ROUND(o.search_radius_deg * 3600.0, 4) AS search_radius_arcsec
    FROM observations AS o
    JOIN run_observations AS ro USING (eso_dp_id)
    LEFT JOIN observation_objects AS oo USING (eso_dp_id)
    WHERE ro.run_id = ?
      AND oo.eso_dp_id IS NULL
    ORDER BY o.eso_dp_id
    """,
    connection,
    params=(RUN_ID,),
)
print(f"Spectra with no SIMBAD match: {len(unmatched)}")
display(unmatched)

## 10. Validate the stored links

All three values below should be zero. The database primary keys prevent duplicates, while the last check confirms that every link lies within the cone used for its ESO spectrum.

In [ ]:
validation = pd.read_sql_query(
    """
    SELECT 'duplicate catalog object keys' AS check_name, COUNT(*) AS failures
    FROM (
        SELECT catalog, catalog_object_id
        FROM catalog_objects
        GROUP BY catalog, catalog_object_id
        HAVING COUNT(*) > 1
    )
    UNION ALL
    SELECT 'duplicate observation-object links', COUNT(*)
    FROM (
        SELECT eso_dp_id, catalog, catalog_object_id
        FROM observation_objects
        GROUP BY eso_dp_id, catalog, catalog_object_id
        HAVING COUNT(*) > 1
    )
    UNION ALL
    SELECT 'links outside search radius', COUNT(*)
    FROM observation_objects AS oo
    JOIN run_observations AS ro USING (eso_dp_id)
    JOIN observations AS o USING (eso_dp_id)
    WHERE ro.run_id = ?
      AND oo.separation_arcsec > o.search_radius_deg * 3600.0 + 1e-6
    """,
    connection,
    params=(RUN_ID,),
)
display(validation)
assert (validation["failures"] == 0).all(), "One or more database checks failed"

## 11. Read the scale estimate carefully

The estimate extrapolates the observed fraction of unique search positions to two million spectra. The SIMBAD number is a batch-count estimate, not a measured production runtime.

In [ ]:
scale_metrics = {
    key: value for key, value in run_summary.items() if key.startswith("scale.")
}
display(pd.Series(scale_metrics, name="value").rename_axis("metric").to_frame())

connection.close()